# Databricks Secret Management Utilities

This module provides helper functions to manage **Databricks Secret Scopes** and **Secrets** programmatically using the Databricks REST API.  
Secrets are used to securely store credentials, tokens, and other sensitive information in Databricks.

---

## ⚙️ Functions Overview

### 1. `create_scope(scope_name: str, backend_type: str)`
Creates a new secret scope in Databricks.

- **Parameters:**
  - `scope_name` → Name of the secret scope to be created.
  - `backend_type` → Type of backend (currently not used in payload, but can be extended for different backends).

- **Process:**
  - Retrieves Databricks instance URL and API token from the notebook context.
  - Calls the Databricks REST API endpoint:  
    `POST /api/2.0/secrets/scopes/create`
  - Sends the scope name in the request payload.

- **Output:**
  - Prints success message if scope is created.
  - Prints error details if creation fails.

---

### 2. `delete_scope(scope_name: str)`
Deletes an existing secret scope.

- **Parameters:**
  - `scope_name` → Name of the secret scope to delete.

- **Process:**
  - Retrieves Databricks instance URL and API token from the notebook context.
  - Calls the Databricks REST API endpoint:  
    `POST /api/2.0/secrets/scopes/delete`
  - Sends the scope name in the request payload.

- **Output:**
  - Prints success message if scope is deleted.
  - Prints error details if deletion fails.

---

### 3. `create_secret(scope_name: str, secret_key: str, secret_value: str)`
Creates or updates a secret inside a given scope.

- **Parameters:**
  - `scope_name` → Name of the secret scope where the secret will be stored.
  - `secret_key` → Key (identifier) for the secret.
  - `secret_value` → Value of the secret (e.g., password, token).

- **Process:**
  - Retrieves Databricks instance URL and API token from the notebook context.
  - Calls the Databricks REST API endpoint:  
    `POST /api/2.0/secrets/put`
  - Sends scope name, key, and secret value in the request payload.

- **Output:**
  - Prints success message if secret is created/updated.
  - Prints error details if operation fails.

---

### 4. `list_secrets(scope_name: str)`
Lists all secrets inside a given scope.

- **Parameters:**
  - `scope_name` → Name of the secret scope to list secrets from.

- **Process:**
  - Retrieves Databricks instance URL and API token from the notebook context.
  - Calls the Databricks REST API endpoint:  
    `GET /api/2.0/secrets/list`
  - Sends the scope name as a query parameter.

- **Output:**
  - Prints all secret keys available in the scope.
  - Note: Only **keys** are returned, not secret values (for security reasons).

---

### 5. `list_scopes()`
Lists all secret scopes available in the Databricks workspace.

- **Process:**
  - Retrieves Databricks instance URL and API token from the notebook context.
  - Calls the Databricks REST API endpoint:  
    `GET /api/2.0/secrets/scopes/list`

- **Output:**
  - Prints all secret scopes available in the workspace.

---

## 📌 Usage Example

```python
# Create a new secret scope
create_scope("my_scope", "DATABRICKS")

# Add a secret to the scope
create_secret("my_scope", "db_password", "SuperSecret123!")

# List all secrets in the scope
list_secrets("my_scope")

# List all secret scopes
list_scopes()

# Delete the scope
delete_scope("my_scope")


In [0]:
import requests
import json

In [0]:
#take scope name and backend_type and created a sceret scope
def create_scope(scope_name: str, backend_type: str):
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    DATABRICKS_INSTANCE= ctx.apiUrl().getOrElse(None)     # e.g. https://adb-...azuredatabricks.net
    DATABRICKS_TOKEN = ctx.apiToken().getOrElse(None)  # personal access token for this session
    url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/scopes/create"
    payload = {"scope": scope_name}
    headers = {
        "Authorization": f"Bearer {DATABRICKS_TOKEN}",
        "Content-Type": "application/json",
    }
    response = requests.post(url, headers=headers, data=json.dumps(payload))
    if response.status_code == 200:
        print(f"Secret scope '{scope_name}' created successfully.")
    else:
        print("Failed to create secret scope.")
        print("Status Code:", response.status_code)
        print("Response:", response.text)

In [0]:
#take scope name  and delete a sceret scope
def delete_scope(scope_name: str):
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    DATABRICKS_INSTANCE= ctx.apiUrl().getOrElse(None)     # e.g. https://adb-...azuredatabricks.net
    DATABRICKS_TOKEN = ctx.apiToken().getOrElse(None)  # personal access token for this session
    url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/scopes/delete"
    payload = {"scope": scope_name}
    headers = {
        "Authorization": f"Bearer {DATABRICKS_TOKEN}",
        "Content-Type": "application/json",
    }
    response = requests.post(url, headers=headers, data=json.dumps(payload))
    if response.status_code == 200:
        print(f"Secret scope '{scope_name}' deleted successfully.")
    else:
        print("Failed to delete secret scope.")
        print("Status Code:", response.status_code)
        print("Response:", response.text)

In [0]:
def create_secret(scope_name: str, secret_key: str, secret_value: str):
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    DATABRICKS_INSTANCE = ctx.apiUrl().getOrElse(None)
    DATABRICKS_TOKEN = ctx.apiToken().getOrElse(
        None
    )  # personal access token for this session
    url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/put"
    headers = {
        "Authorization": f"Bearer {DATABRICKS_TOKEN}",
        "Content-Type": "application/json",
    }
    payload = {"scope": scope_name, "key": secret_key, "string_value": secret_value}
    response = requests.post(url, headers=headers, data=json.dumps(payload))
    if response.status_code == 200:
        print(f"Secret '{secret_key}' created successfully in scope '{scope_name}'.")
    else:
        print("Failed to create secret.")
        print("Status:", response.status_code)
        print("Response:", response.text)

In [0]:
def read_secret(scope_name: str, secret_key: str):
    try:
        retrieved_json = dbutils.secrets.get(
            scope=scope_name,
            key=secret_key
        )
        print("Secret retrieved successfully.")
        parsed = json.loads(retrieved_json)
        print("Parsed JSON:")
        print(parsed)
    except Exception as e:
        print("Failed to retrieve or parse secret.")
        print("Error:", str(e))

In [0]:
def list_secrets(scope_name: str):
    # Get Databricks context
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    DATABRICKS_INSTANCE = ctx.apiUrl().getOrElse(None)
    DATABRICKS_TOKEN = ctx.apiToken().getOrElse(None)

    # API endpoint
    url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/list"
    headers = {
        "Authorization": f"Bearer {DATABRICKS_TOKEN}",
        "Content-Type": "application/json",
    }
    payload = {"scope": scope_name}

    # Call API
    response = requests.get(url, headers=headers, params=payload)

    if response.status_code == 200:
        secrets = response.json().get("secrets", [])
        print(f"Secrets in scope '{scope_name}':")
        for secret in secrets:
            print(f"- Key: {secret['key']}")
    else:
        print("Failed to list secrets.")
        print("Status Code:", response.status_code)
        print("Response:", response.text)


In [0]:
def list_scopes():
    # Get Databricks context
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    DATABRICKS_INSTANCE = ctx.apiUrl().getOrElse(None)
    DATABRICKS_TOKEN = ctx.apiToken().getOrElse(None)

    # API endpoint
    url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/scopes/list"
    headers = {
        "Authorization": f"Bearer {DATABRICKS_TOKEN}",
        "Content-Type": "application/json",
    }

    # Call API
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        scopes = response.json().get("scopes", [])
        print("Available secret scopes:")
        for scope in scopes:
            print(f"- Scope: {scope['name']}")
    else:
        print("Failed to list scopes.")
        print("Status Code:", response.status_code)
        print("Response:", response.text)
